# SMART Dataset — Exploratory Data Analysis

**Dataset:** 175 ICU patient time series files (long format: one row per measurement)  
**Columns per file:** `Feature`, `Value`, `Unit`, `Time`  
**Goal:** Understand data quality, coverage, and feature distributions before modeling.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

DATA_DIR = Path('data')
patient_files = sorted(DATA_DIR.glob('ID_*.csv'))
print(f'Found {len(patient_files)} patient files')

## 1. Load all patient data

In [ ]:
all_dfs = []
load_errors = []

for f in patient_files:
    try:
        df = pd.read_csv(f, parse_dates=['Time'], encoding='latin-1')
        df['patient_id'] = f.stem  # e.g. 'ID_001'
        all_dfs.append(df)
    except Exception as e:
        load_errors.append((f.stem, str(e)))

if load_errors:
    print('Files that failed to load:')
    for pid, err in load_errors:
        print(f'  {pid}: {err}')

full_df = pd.concat(all_dfs, ignore_index=True)
print(f'Total rows: {len(full_df):,}')
print(f'Columns: {list(full_df.columns)}')
full_df.head()

## 2. Per-patient summary: row count, time span, date range

In [ ]:
patient_summary = full_df.groupby('patient_id').agg(
    n_rows=('Feature', 'count'),
    n_unique_features=('Feature', 'nunique'),
    first_timestamp=('Time', 'min'),
    last_timestamp=('Time', 'max'),
).reset_index()

patient_summary['duration_days'] = (
    patient_summary['last_timestamp'] - patient_summary['first_timestamp']
).dt.total_seconds() / 86400

patient_summary['year'] = patient_summary['first_timestamp'].dt.year

print(patient_summary.describe())
patient_summary.sort_values('n_rows').head(10)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].hist(patient_summary['n_rows'], bins=30, color='steelblue', edgecolor='white')
axes[0].set_title('Rows per patient')
axes[0].set_xlabel('Number of measurement rows')
axes[0].set_ylabel('Patient count')

axes[1].hist(patient_summary['duration_days'].dropna(), bins=30, color='coral', edgecolor='white')
axes[1].set_title('Monitoring duration per patient (days)')
axes[1].set_xlabel('Days')

axes[2].hist(patient_summary['n_unique_features'], bins=20, color='seagreen', edgecolor='white')
axes[2].set_title('Unique features per patient')
axes[2].set_xlabel('Number of distinct features')

plt.tight_layout()
plt.savefig('data/fig_patient_overview.png', dpi=120)
plt.show()

In [ ]:
# Flag patients with very few rows or very short duration
LOW_ROW_THRESHOLD = 100
SHORT_DURATION_THRESHOLD = 1  # days

sparse_patients = patient_summary[
    (patient_summary['n_rows'] < LOW_ROW_THRESHOLD) |
    (patient_summary['duration_days'] < SHORT_DURATION_THRESHOLD)
].sort_values('n_rows')

print(f'Patients with <{LOW_ROW_THRESHOLD} rows OR <{SHORT_DURATION_THRESHOLD} day duration: {len(sparse_patients)}')
sparse_patients[['patient_id', 'n_rows', 'duration_days', 'n_unique_features', 'year']]

## 3. Feature coverage across patients (using Feature_File_Count_Summary.csv)

In [ ]:
feat_summary = pd.read_csv(DATA_DIR / 'Feature_File_Count_Summary.csv')
feat_summary.columns = feat_summary.columns.str.strip()
feat_summary = feat_summary.sort_values('File_Count', ascending=False).reset_index(drop=True)

n_patients = len(patient_files)
feat_summary['coverage_pct'] = feat_summary['File_Count'] / n_patients * 100

print(f'Total unique features: {len(feat_summary)}')
print(f'Features present in >90% of patients: {(feat_summary["coverage_pct"] > 90).sum()}')
print(f'Features present in <50% of patients: {(feat_summary["coverage_pct"] < 50).sum()}')

feat_summary.head(20)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(feat_summary['coverage_pct'], bins=20, color='mediumpurple', edgecolor='white')
ax.set_xlabel('% of patients with this feature')
ax.set_ylabel('Number of features')
ax.set_title('Feature coverage distribution across 175 patients')
ax.axvline(90, color='red', linestyle='--', label='90% threshold')
ax.axvline(50, color='orange', linestyle='--', label='50% threshold')
ax.legend()
plt.tight_layout()
plt.savefig('data/fig_feature_coverage.png', dpi=120)
plt.show()

In [ ]:
# Top 30 most common features
top30 = feat_summary.head(30)

fig, ax = plt.subplots(figsize=(10, 9))
bars = ax.barh(top30['Feature'][::-1], top30['coverage_pct'][::-1], color='steelblue')
ax.set_xlabel('% of patients')
ax.set_title('Top 30 features by patient coverage')
ax.axvline(90, color='red', linestyle='--', alpha=0.6, label='90%')
ax.legend()
plt.tight_layout()
plt.savefig('data/fig_top30_features.png', dpi=120)
plt.show()

## 4. Value distribution and outlier detection for key features

In [ ]:
# Coerce Value column to numeric (some might be strings/errors)
full_df['Value_num'] = pd.to_numeric(full_df['Value'], errors='coerce')

non_numeric_rate = full_df['Value_num'].isna().mean() * 100
print(f'Non-numeric value rate: {non_numeric_rate:.2f}%')

# Check what non-numeric values look like
non_numeric_examples = full_df[full_df['Value_num'].isna()]['Value'].value_counts().head(20)
print('\nMost common non-numeric values:')
print(non_numeric_examples)

In [ ]:
# Distribution stats for all features
feature_stats = full_df.groupby('Feature')['Value_num'].agg(
    count='count',
    mean='mean',
    std='std',
    min='min',
    p25=lambda x: x.quantile(0.25),
    median='median',
    p75=lambda x: x.quantile(0.75),
    max='max',
    n_missing=lambda x: x.isna().sum(),
).reset_index()

feature_stats['missing_pct'] = feature_stats['n_missing'] / len(full_df) * 100
feature_stats = feature_stats.sort_values('count', ascending=False)
print('Feature value statistics (top 20 by count):')
feature_stats.head(20)

In [ ]:
# Plot value distributions for top 12 most frequent features
top_features = feature_stats.head(12)['Feature'].tolist()

fig, axes = plt.subplots(3, 4, figsize=(18, 12))
axes = axes.flatten()

for i, feat in enumerate(top_features):
    vals = full_df[full_df['Feature'] == feat]['Value_num'].dropna()
    axes[i].hist(vals, bins=40, color='steelblue', edgecolor='white')
    unit = full_df[full_df['Feature'] == feat]['Unit'].dropna().mode()
    unit_str = unit.iloc[0] if len(unit) > 0 else ''
    axes[i].set_title(f'{feat}\n[{unit_str}]  n={len(vals):,}', fontsize=8)
    axes[i].set_xlabel('Value', fontsize=7)

plt.suptitle('Value distributions — top 12 features', y=1.01, fontsize=12)
plt.tight_layout()
plt.savefig('data/fig_value_distributions.png', dpi=120, bbox_inches='tight')
plt.show()

## 5. Outlier detection (IQR method)

In [ ]:
def count_outliers_iqr(series, factor=3.0):
    q1, q3 = series.quantile(0.25), series.quantile(0.75)
    iqr = q3 - q1
    lower, upper = q1 - factor * iqr, q3 + factor * iqr
    n_out = ((series < lower) | (series > upper)).sum()
    return n_out, round(n_out / len(series) * 100, 2), lower, upper

outlier_report = []
for feat in feature_stats['Feature']:
    vals = full_df[full_df['Feature'] == feat]['Value_num'].dropna()
    if len(vals) < 10:
        continue
    n_out, pct_out, lower, upper = count_outliers_iqr(vals)
    outlier_report.append({
        'Feature': feat,
        'n_values': len(vals),
        'n_outliers': n_out,
        'outlier_pct': pct_out,
        'iqr_lower': round(lower, 3),
        'iqr_upper': round(upper, 3),
        'actual_min': round(vals.min(), 3),
        'actual_max': round(vals.max(), 3),
    })

outlier_df = pd.DataFrame(outlier_report).sort_values('outlier_pct', ascending=False)
print('Features with highest outlier rates (IQR x3):')
outlier_df.head(20)

## 6. Temporal coverage — admission years and monitoring timelines

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Year of first admission
patient_summary['year'].value_counts().sort_index().plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Patient admission year distribution')
axes[0].set_xlabel('Year')
axes[0].set_ylabel('Number of patients')

# Monitoring duration scatter
axes[1].scatter(range(len(patient_summary)), patient_summary.sort_values('duration_days')['duration_days'],
               alpha=0.5, color='coral', s=20)
axes[1].set_title('Monitoring duration per patient (sorted)')
axes[1].set_xlabel('Patient rank')
axes[1].set_ylabel('Days')
axes[1].axhline(1, color='red', linestyle='--', label='1 day')
axes[1].legend()

plt.tight_layout()
plt.savefig('data/fig_temporal_coverage.png', dpi=120)
plt.show()

## 7. Missing feature matrix — which features are present for which patients?

In [ ]:
# Use only top 40 features for readability
top40_features = feat_summary.head(40)['Feature'].tolist()

presence_matrix = full_df[full_df['Feature'].isin(top40_features)].groupby(
    ['patient_id', 'Feature']
).size().unstack(fill_value=0)

presence_binary = (presence_matrix > 0).astype(int)

fig, ax = plt.subplots(figsize=(16, 10))
sns.heatmap(presence_binary.T, cmap='Blues', linewidths=0, ax=ax,
            cbar_kws={'label': 'Feature present'}, xticklabels=False)
ax.set_title('Feature presence matrix — top 40 features x 175 patients', fontsize=11)
ax.set_ylabel('Feature')
ax.set_xlabel('Patient (sorted by ID)')
plt.tight_layout()
plt.savefig('data/fig_feature_presence_matrix.png', dpi=120)
plt.show()

## 8. Export summary tables for the meeting

In [ ]:
patient_summary.to_csv('data/eda_patient_summary.csv', index=False)
feature_stats.to_csv('data/eda_feature_stats.csv', index=False)
outlier_df.to_csv('data/eda_outlier_report.csv', index=False)

print('Saved:')
print('  data/eda_patient_summary.csv  — one row per patient')
print('  data/eda_feature_stats.csv    — value stats per feature')
print('  data/eda_outlier_report.csv   — outlier rates per feature')
print('  data/fig_*.png                — all plots')

## 9. Key findings summary (fill in after running)

| Question | Finding |
|---|---|
| Total patients | 175 |
| Patients with very few rows (<100) | _fill in_ |
| Patients with very short duration (<1 day) | _fill in_ |
| Total unique features | _fill in_ |
| Features in >90% patients | _fill in_ |
| Features in <50% patients | _fill in_ |
| Non-numeric value rate | _fill in_ |
| Features with highest outlier % | _fill in_ |